# 🔍 SQL Murder Mystery — My Investigation

A step-by-step SQL investigation solving the [SQL Murder Mystery](https://mystery.knightlab.com/), using the dataset from [Kaggle](https://www.kaggle.com/datasets/johnp47/sql-murder-mystery-database).

**Goal:** find out who committed the murder that took place on **January 15, 2018** in **SQL City**, using nothing but SQL queries against the provided database.

## How to run this yourself
This notebook connects to the included `sql-murder-mystery.db` SQLite file — no server, no setup, no credentials needed.

- **Run locally:** clone this repo, open this notebook in Jupyter (`pip install notebook pandas` if needed), and run all cells.
- **Run in your browser with zero installs:** click the "Open in Colab" badge at the top of the README, upload/keep this notebook + the `.db` file in the same session folder, and run all cells.

Each step below shows: the clue I was chasing, the SQL query I ran, and the actual result.


In [1]:
# Setup: connect to the SQLite database
import sqlite3
import pandas as pd

conn = sqlite3.connect("sql-murder-mystery.db")

def q(sql):
    """Run a SQL query and return the results as a pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

# Quick sanity check: list all tables
q("SELECT name FROM sqlite_master WHERE type='table';")


,name
0,crime_scene_report
1,drivers_license
2,person
3,facebook_event_checkin
4,interview
5,get_fit_now_member
6,get_fit_now_check_in
7,income
8,solution


## Step 1 — Find the murder report

The case always starts with a report. Let's look for a `murder` report in `crime_scene_report`.


In [2]:
q('''
SELECT *
FROM crime_scene_report
WHERE type = 'murder'
  AND city = 'SQL City';
''')


,date,type,description,city
0,20180215,murder,REDACTED REDACTED REDACTED,SQL City
1,20180215,murder,Someone killed the guard! He took an arrow to ...,SQL City
2,20180115,murder,Security footage shows that there were 2 witne...,SQL City


**Result:** among the murder reports, one matches our case — dated **2018-01-15**, in SQL City:

> *"Security footage shows that there were 2 witnesses. The first witness lives at the last house on 'Northwestern Dr'. The second witness, named Annabel, lives somewhere on 'Franklin Ave'."*

That gives us two leads: a witness at the highest house number on Northwestern Dr, and a witness named Annabel on Franklin Ave.


## Step 2 — Identify the two witnesses

**Witness 1:** the *last* house on Northwestern Dr → highest `address_number` on that street.


In [3]:
q('''
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 1;
''')


,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


**Witness 2:** named Annabel, lives on Franklin Ave.


In [4]:
q('''
SELECT *
FROM person
WHERE name LIKE 'Annabel%'
  AND address_street_name = 'Franklin Ave';
''')


,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


**Result:** our two witnesses are **Morty Schapiro** (Northwestern Dr) and **Annabel Miller** (Franklin Ave).

Next step: pull their statements from the `interview` table.


## Step 3 — Read their interview transcripts

Now that we have both `person.id` values, let's join into `interview` to see what they told investigators.


In [5]:
q('''
SELECT p.name, i.transcript
FROM interview i
JOIN person p ON p.id = i.person_id
WHERE p.name IN ('Morty Schapiro', 'Annabel Miller');
''')


,name,transcript
0,Morty Schapiro,I heard a gunshot and then saw a man run out. ...
1,Annabel Miller,"I saw the murder happen, and I recognized the ..."


👉 **Your turn from here.** Read what the witnesses said, and follow the trail — you'll likely end up cross-referencing:
- `get_fit_now_member` + `get_fit_now_check_in` (if a gym membership is mentioned)
- `drivers_license` (if a physical description, plate number, or car is mentioned)
- `facebook_event_checkin` (if an event comes up)
- `income` (to corroborate a suspect's profile)

Keep adding a markdown cell (your reasoning) + a code cell (your query) for each step, just like above.


## Final Answer

*(Fill this in once you've solved it — name the culprit and briefly summarize the chain of evidence that led you there.)*


In [6]:
# Once you're confident, you can check your answer against the `solution` table:
q('''
SELECT * FROM solution;
''')
# Note: this table is empty until you INSERT your guess — see the original SQL Murder Mystery site for the exact check format.


,user,value
